# Photovoltaic Thermography Training Template
Baseline notebook for segmentation and fault detection experiments on the Photovoltaic System Thermography dataset.
The dataset archive is stored in Google Drive as `Photovoltaic_dataset.zip`.


## 0. Environment Setup
Set the execution context (Colab or local) and prepare paths.


In [1]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / '.git').exists():
            return parent
    return start

try:
    import google.colab  # type: ignore
    ENV = 'colab'
except ModuleNotFoundError:
    ENV = 'local'

if ENV == 'colab':
    PROJECT_ROOT = Path('/content/hafar-pv-maintenance')
    DATA_ROOT = Path('/content/data/photovoltaic-system-thermography')
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = find_repo_root(cwd)
    DATA_ROOT = PROJECT_ROOT / 'data' / 'photovoltaic-system-thermography'
print(f'Running in {ENV} mode')
print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')


Running in local mode
Project root: e:\Fajr Project\hafar-pv-maintenance\notebooks
Data root: e:\Fajr Project\hafar-pv-maintenance\notebooks\data\photovoltaic-system-thermography


### Drive Mount (Colab only)
Connect Google Drive when running on Colab so the dataset ZIP is available.


In [2]:
from pathlib import Path

if ENV == 'colab':
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_DATA_DIR = Path('/content/drive/MyDrive/Photovoltaic_dataset')
else:
    DRIVE_DATA_DIR = None
print(f'Drive data directory: {DRIVE_DATA_DIR}')


Drive data directory: None


## 1. Dependencies
Install or upgrade project requirements if the environment does not already contain them.

In [3]:
if ENV == 'colab':
    !pip install --quiet flirimageextractor flyr albumentations segmentation-models-pytorch pytorch-lightning wandb seaborn pydantic-settings
else:
    !pip install -e .


Install project dependencies via pip if they are not already available in this environment.


## 2. Data Access
Extract the Drive-hosted archive into the runtime and point `DATASET_SELECTION` at the desired dataset folder.
The default configuration expects `Photovoltaic_dataset.zip` within `MyDrive/Photovoltaic_dataset`.


In [8]:
from pathlib import Path
import zipfile

DATASET_ZIP_NAME = 'Photovoltaic_dataset.zip'  # update if the archive name differs
DATASET_SELECTION = 0  # choose which extracted dataset folder to use

data_root = Path(DATA_ROOT)
dataset_candidates: list[Path] = []

if ENV == 'colab':
    if DRIVE_DATA_DIR is None:
        raise FileNotFoundError('Drive mount is missing; rerun the previous cell.')
    dataset_zip = DRIVE_DATA_DIR / DATASET_ZIP_NAME
    if not dataset_zip.exists():
        raise FileNotFoundError(f'Expected archive not found: {dataset_zip}')
    extract_root = Path('/content/data/raw_photovoltaic')
    extract_root.mkdir(parents=True, exist_ok=True)
    sentinel = extract_root / '.extracted'
    if not sentinel.exists():
        print(f'Extracting {dataset_zip} -> {extract_root}')
        with zipfile.ZipFile(dataset_zip) as zf:
            zf.extractall(extract_root)
        sentinel.write_text('extracted')
    dataset_candidates = [candidate for candidate in extract_root.iterdir() if candidate.is_dir()]
else:
    search_roots = []
    local_raw = Path(PROJECT_ROOT) / 'data' / 'raw'
    search_roots.append(local_raw)
    search_roots.append(Path.cwd().resolve() / 'data' / 'raw')
    search_roots.append(Path.cwd().resolve().parent / 'data' / 'raw')
    for root in search_roots:
        if root.exists():
            dataset_candidates.extend(p for p in root.iterdir() if p.is_dir())
    if not dataset_candidates and data_root.exists():
        dataset_candidates = [data_root]

json_candidates = [candidate for candidate in dataset_candidates if any(candidate.rglob('*.json'))]
if json_candidates:
    dataset_candidates = json_candidates

if not dataset_candidates:
    raise FileNotFoundError('Could not locate any dataset directories; verify the archive contents.')

if DATASET_SELECTION >= len(dataset_candidates):
    raise IndexError('DATASET_SELECTION is out of range; choose a smaller index.')

for idx, candidate in enumerate(dataset_candidates):
    marker = '->' if idx == DATASET_SELECTION else '  '
    print(f'{marker} [{idx}] {candidate}')

data_root = dataset_candidates[DATASET_SELECTION]
print(f'Resolved data root: {data_root}')
if not data_root.exists():
    raise FileNotFoundError('Selected dataset directory does not exist; check DATASET_SELECTION.')


-> [0] E:\Fajr Project\hafar-pv-maintenance\data\raw\dataset_1
   [1] E:\Fajr Project\hafar-pv-maintenance\data\raw\dataset_2
Resolved data root: E:\Fajr Project\hafar-pv-maintenance\data\raw\dataset_1


### Repository Checkout (Colab)
Clone the project from GitHub and install it so shared modules are importable before running EDA.


In [9]:
REPO_URL = 'https://github.com/mohammedradman1/hafar-pv-maintenance.git'

if ENV == 'colab':
    if PROJECT_ROOT.exists():
        %cd $PROJECT_ROOT
        !git pull
    else:
        !git clone {REPO_URL} {PROJECT_ROOT}
        %cd $PROJECT_ROOT
    !pip install --quiet -e .
else:
    print('Running outside Colab; ensure the project is available locally.')


Running outside Colab; ensure the project is available locally.


## 2a. Exploratory Data Analysis
Inspect dataset structure, label balance, and representative samples before training.

Planned steps for EDA (we will execute these one by one):
1) Load dataset metadata (manifests) and report basic counts.
2) Scene-level summary: modules per image, defect counts, dimensions.
3) Panel-level metrics: area, aspect ratio, temperature stats, label balance.
4) Qualitative checks: overlay polygons on scenes and view sample crops.


In [ ]:
# EDA imports and path setup
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

repo_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
src_path = repo_root / 'src'
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from hafar_pv.data.photovoltaic_thermography import (
    PhotovoltaicThermographyPreprocessor,
    load_radiometric_frame,
)

sns.set_theme(style='whitegrid')


In [ ]:
# Load manifests; if missing, run preprocessing using data_root
from pathlib import Path

if 'data_root' not in globals():
    raise RuntimeError('data_root is undefined; run the Data Access cell first.')

processed_root = repo_root / 'data' / 'processed' / 'photovoltaic-system-thermography'
meta_dir = processed_root / 'metadata'
panels_manifest_path = meta_dir / 'panels_manifest.csv'
scenes_manifest_path = meta_dir / 'scenes_manifest.csv'

def _load_manifests():
    pm = pd.read_csv(panels_manifest_path)
    pm['npz_path'] = pm['npz_path'].apply(lambda p: processed_root / Path(p))
    sm = pd.read_csv(scenes_manifest_path)
    sm['npz_path'] = sm['npz_path'].apply(lambda p: processed_root / Path(p))
    return pm, sm

if panels_manifest_path.exists() and scenes_manifest_path.exists():
    panels_manifest, scenes_manifest = _load_manifests()
    print('Loaded manifests from', meta_dir)
else:
    meta_dir.mkdir(parents=True, exist_ok=True)
    print('Manifests not found; running preprocessing to generate them...')
    preprocessor = PhotovoltaicThermographyPreprocessor(
        raw_root=Path(data_root),
        output_root=processed_root,
        resize_panels_to=(224, 224),
        val_fraction=0.2,
        test_fraction=0.1,
    )
    preprocessor.run()
    if panels_manifest_path.exists() and scenes_manifest_path.exists():
        panels_manifest, scenes_manifest = _load_manifests()
        print('Preprocessing complete; manifests loaded.')
    else:
        raise RuntimeError('Preprocessing did not produce manifests; check raw data layout.')


In [ ]:
# Basic counts from manifests
scene_df = scenes_manifest.copy()
panel_df = panels_manifest.copy()

print(f'Scenes: {len(scene_df)}')
print(f'Panels: {len(panel_df)}')
if 'split' in scene_df.columns:
    print('Scene splits:', scene_df['split'].value_counts().to_dict())
if 'split' in panel_df.columns:
    print('Panel splits:', panel_df['split'].value_counts().to_dict())
if 'defective' in panel_df.columns:
    print('Label balance:', panel_df['defective'].value_counts().to_dict())

scene_df.head()


In [ ]:
# Scene-level summary
scene_metrics = scene_df[['num_modules', 'num_defective', 'temperature_mean', 'temperature_max']].describe()
print(scene_metrics)

scene_df[['num_modules', 'num_defective']].hist(figsize=(10, 4))
plt.tight_layout()


In [ ]:
# Panel-level metrics
panel_cols = ['area_px', 'bbox_width', 'bbox_height', 'aspect_ratio', 'mean_temperature', 'max_temperature']
available_cols = [c for c in panel_cols if c in panel_df.columns]
panel_stats = panel_df[available_cols].describe(percentiles=[0.1, 0.5, 0.9])
print(panel_stats)


In [ ]:
# Distributions: area, aspect ratio, label balance
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
if 'area_px' in panel_df.columns:
    sns.histplot(panel_df['area_px'], ax=axes[0], bins=30, color='#1f77b4')
    axes[0].set_title('Module Pixel Area')
if 'aspect_ratio' in panel_df.columns:
    sns.histplot(panel_df['aspect_ratio'].dropna(), ax=axes[1], bins=30, color='#ff7f0e')
    axes[1].set_title('Module Aspect Ratio')
if 'defective' in panel_df.columns:
    sns.countplot(x='defective', data=panel_df, ax=axes[2], palette='Set2')
    axes[2].set_title('Defect Label Balance')
    axes[2].set_xticklabels(['Healthy', 'Defective'])
plt.tight_layout()


In [ ]:
# Qualitative check: overlay polygons on a sample scene
preprocessor = PhotovoltaicThermographyPreprocessor(
    raw_root=Path(data_root),
    output_root=processed_root,
    resize_panels_to=(224, 224),
)
scenes = preprocessor._discover_scenes()
if not scenes:
    raise RuntimeError('No scenes found; verify raw data.')

def plot_scene_with_polygons(scene):
    frame = load_radiometric_frame(scene.image_path)
    normalized = (frame - frame.min()) / (frame.ptp() + 1e-6)
    plt.figure(figsize=(6, 6))
    plt.imshow(normalized, cmap='inferno')
    for module in scene.modules:
        polygon = np.asarray(module.polygon, dtype=np.float32)
        xs = np.append(polygon[:, 0], polygon[0, 0])
        ys = np.append(polygon[:, 1], polygon[0, 1])
        color = 'red' if module.defective else 'lime'
        plt.plot(xs, ys, color=color, linewidth=1.5)
    plt.title(f'Scene {scene.image_path.stem} ({len(scene.modules)} modules)')
    plt.axis('off')
    plt.tight_layout()

plot_scene_with_polygons(scenes[0])


## 3. Project Checkout
Clone or update the project repository so we can import reusable modules.

In [ ]:
project_root = Path(PROJECT_ROOT)
if ENV == 'colab':
    print('Repository cloned above for Colab; skipping new checkout.')
else:
    if not project_root.exists():
        !git clone https://github.com/your-org/hafar-pv-maintenance $PROJECT_ROOT
    else:
        print('Repository already present, skipping clone.')
    %cd $PROJECT_ROOT
    !pip install --quiet -e .[dev]


## 4. Configuration & Logging
Load application settings and initialize experiment tracking (optional).

In [ ]:
from hafar_pv.config import get_settings
from hafar_pv.utils.logging import configure_logging

settings = get_settings()
configure_logging()
print(settings.json(indent=2))
# Optional: import wandb and start a run here.

## 5. Preprocessing Review
If processed artifacts do not exist yet, run the radiometric extraction and mask generation utilities.

In [ ]:
from hafar_pv.data import PhotovoltaicThermographyPreprocessor

processed_root = Path(settings.data_root) / 'processed' / 'photovoltaic-system-thermography'
metadata_dir = processed_root / 'metadata'
manifest_path = metadata_dir / 'panels_manifest.csv'

if not manifest_path.exists():
    candidate_roots = [data_root] + [p for p in data_root.iterdir() if p.is_dir()]
    raw_root = None
    for candidate in candidate_roots:
        if list(candidate.rglob('*.json')):
            raw_root = candidate
            break
    if raw_root is None:
        raise FileNotFoundError('Could not locate annotation JSON files. Please unzip the dataset.')
    print(f'Running preprocessing from {raw_root} -> {processed_root}')
    preprocessor = PhotovoltaicThermographyPreprocessor(
        raw_root=raw_root,
        output_root=processed_root,
        resize_panels_to=(224, 224),
        val_fraction=0.2,
        test_fraction=0.1,
    )
    preprocessor.run()
else:
    print('Processed artifacts found; skipping preprocessing.')

In [ ]:
from hafar_pv.data import load_panels_manifest, load_scenes_manifest

processed_root = Path(settings.data_root) / 'processed' / 'photovoltaic-system-thermography'
panels_manifest = load_panels_manifest(processed_root)
scenes_manifest = load_scenes_manifest(processed_root)
scene_split = (
    panels_manifest.groupby('scene_id')['split']
    .agg(lambda s: s.value_counts().idxmax())
    .rename('split')
)
scenes_manifest = scenes_manifest.merge(scene_split, on='scene_id', how='left')
scenes_manifest['split'] = scenes_manifest['split'].fillna('train')
print('Panel split counts:', panels_manifest['split'].value_counts().to_dict())
print('Scene split counts:', scenes_manifest['split'].value_counts().to_dict())

## 6. Segmentation Training
Train a segmentation model on thermal frames and corresponding module masks.

In [ ]:
from hafar_pv.segmentation.train import SegmentationTrainer, SegmentationConfig
from hafar_pv.data.datasets import PanelDataset

train_scene_paths = scenes_manifest[scenes_manifest['split'] == 'train']['npz_path'].tolist()
val_scene_paths = scenes_manifest[scenes_manifest['split'] == 'val']['npz_path'].tolist()
if not val_scene_paths:
    val_scene_paths = scenes_manifest[scenes_manifest['split'] == 'test']['npz_path'].tolist()
train_ds = PanelDataset([Path(p) for p in train_scene_paths])
val_ds = PanelDataset([Path(p) for p in val_scene_paths]) if val_scene_paths else None
print(f'Segmentation train scenes: {len(train_scene_paths)}, val scenes: {len(val_scene_paths)}')
seg_config = SegmentationConfig(model_name="unet-resnet34", batch_size=4, max_epochs=25)
seg_trainer = SegmentationTrainer(seg_config)
# seg_trainer.fit(train_ds, val_ds)

## 7. Segmentation Evaluation
Visualize predictions and compute validation metrics.

In [ ]:
# TODO: load best checkpoint, run inference on validation set, and visualize masks.
print("Add evaluation logic once checkpoints are available.")

## 8. Fault Detection Training
Fine-tune a classifier on module crops labelled as defective or healthy.

In [ ]:
from hafar_pv.faults.train import FaultDetectionTrainer, FaultDetectionConfig
from hafar_pv.data.datasets import PanelDataset

train_cls_paths = panels_manifest[panels_manifest['split'] == 'train']['npz_path'].tolist()
val_cls_paths = panels_manifest[panels_manifest['split'] == 'val']['npz_path'].tolist()
if not val_cls_paths:
    val_cls_paths = panels_manifest[panels_manifest['split'] == 'test']['npz_path'].tolist()
train_cls_ds = PanelDataset([Path(p) for p in train_cls_paths], target_key='label')
val_cls_ds = PanelDataset([Path(p) for p in val_cls_paths], target_key='label') if val_cls_paths else None
num_classes = int(panels_manifest['defective'].nunique())
print(f'Classification train panels: {len(train_cls_paths)}, val panels: {len(val_cls_paths)}')
cls_config = FaultDetectionConfig(model_name="efficientnet_v2_s", batch_size=16, max_epochs=20)
cls_trainer = FaultDetectionTrainer(cls_config)
# cls_trainer.fit(train_cls_ds, val_cls_ds, num_classes=num_classes)

## 9. Fault Detection Evaluation
Compute classification metrics and inspect confusion matrices.

In [ ]:
# TODO: add evaluation including ROC curves, confusion matrix, calibration.
print("Add classification evaluation once checkpoints are available.")

## 10. Export Artifacts
Persist best checkpoints, metrics, and sample outputs for downstream integration.

In [ ]:
from pathlib import Path

artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(exist_ok=True)
summary = {
    'segmentation_checkpoint': 'path/to/segmentation.ckpt',
    'fault_checkpoint': 'path/to/fault.ckpt',
    'metrics': 'metrics.json',
}
with open(artifacts_dir / 'manifest.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, indent=2)
print(f"Artifacts manifest written to {artifacts_dir / 'manifest.json'}")
print("Upload artifacts via Kaggle output panel or copy to Google Drive.")